# Job Change Prediction

## Problem Statement

### Businees Context

In [ ]:
import numpy as np
import pandas as pd

                                           
import matplotlib.pyplot as plt
import seaborn as sns

                                                           
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    recall_score,
    precision_score,
    confusion_matrix,
    roc_auc_score,
    ConfusionMatrixDisplay,
)

                          
from sklearn.impute import SimpleImputer

                                      
from sklearn.linear_model import LogisticRegression

                                    
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

                          
import warnings

warnings.filterwarnings("ignore")

## Loading the dataset

In [ ]:
df = pd.read_csv("jobs_data.csv")

## Data Overview

### Displaying the first few rows of the dataset

In [ ]:
df.head()

### Checking the shape of the dataset

In [ ]:
df.shape

### Checking the data types of the columns for the dataset

In [ ]:
df.info()

### Statistical summary of the dataset

In [ ]:
df.describe(include="all").T

In [ ]:
df.describe(exclude=np.number).T

In [ ]:
cat_col = df.columns

                                                                                    
for column in cat_col:
    print(df[column].value_counts())
    print("-" * 50)

### Checking for duplicate values

In [ ]:
df.duplicated().sum()

### Checking for missing values

In [ ]:
round(df.isnull().sum() / df.isnull().count() * 100, 2)

## Exploratory Data Analysis (EDA) Summary

### Note: The EDA section has been covered multiple times in the previous case studies. In this case study, we will mainly focus on the model building aspects. We will only be looking at the key observations from EDA. The detailed EDA can be found in the appendix section.

In [ ]:
def histogram_boxplot(data, feature, figsize=(12, 7), kde=False, bins=None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (12,7))
    kde: whether to the show density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,                                         
        sharex=True,                                            
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )                           
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )                                                                                     
    sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2
    )                 
    ax_hist2.axvline(
        data[feature].mean(), color="green", linestyle="--"
    )                             
    ax_hist2.axvline(
        data[feature].median(), color="black", linestyle="-"
    )

In [ ]:
def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])                        
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 2, 6))
    else:
        plt.figure(figsize=(n + 2, 6))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        hue=feature,
        palette="Paired",
        order=data[feature].value_counts().index[:n],
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )                                            
        else:
            label = p.get_height()                                       

        x = p.get_x() + p.get_width() / 2                     
        y = p.get_height()                      

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )                           

    plt.show()

In [ ]:
def stacked_barplot(data, predictor, target):
    """
    Print the category counts and plot a stacked bar chart

    data: dataframe
    predictor: independent variable
    target: target variable
    """
    count = data[predictor].nunique()
    sorter = data[target].value_counts().index[-1]
    tab1 = pd.crosstab(data[predictor], data[target], margins=True).sort_values(
        by=sorter, ascending=False
    )
    print(tab1)
    print("-" * 120)
    tab = pd.crosstab(data[predictor], data[target], normalize="index").sort_values(
        by=sorter, ascending=False
    )
    tab.plot(kind="bar", stacked=True, figsize=(count + 1, 5))
    plt.legend(
        loc="lower left", frameon=False,
    )
    plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.show()

In [ ]:
def distribution_plot_wrt_target(data, predictor, target):

    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    target_uniq = data[target].unique()

    axs[0, 0].set_title("Distribution of target for target=" + str(target_uniq[0]))
    sns.histplot(
        data=data[data[target] == target_uniq[0]],
        x=predictor,
        kde=True,
        ax=axs[0, 0],
        color="teal",
    )

    axs[0, 1].set_title("Distribution of target for target=" + str(target_uniq[1]))
    sns.histplot(
        data=data[data[target] == target_uniq[1]],
        x=predictor,
        kde=True,
        ax=axs[0, 1],
        color="orange",
    )

    axs[1, 0].set_title("Boxplot w.r.t target")
    sns.boxplot(data=data, x=target, y=predictor, ax=axs[1, 0], palette="gist_rainbow")

    axs[1, 1].set_title("Boxplot (without outliers) w.r.t target")
    sns.boxplot(
        data=data,
        x=target,
        y=predictor,
        ax=axs[1, 1],
        showfliers=False,
        palette="gist_rainbow",
    )

    plt.tight_layout()
    plt.show()

### Univariate analysis

#### target

In [ ]:
labeled_barplot(df, "target", perc=True)

Most people in this dataset are not looking for a job change (75.1%).

#### city_development_index

In [ ]:
histogram_boxplot(df, "city_development_index")

Most city development index values are on the higher side, with a few lower values.

In [ ]:
df[df["city_development_index"] < 0.45]

#### training_hours

In [ ]:
histogram_boxplot(df, "training_hours")

Training hours are mostly concentrated at lower values, with a few very high values.

#### gender

In [ ]:
labeled_barplot(df, "gender")

Male candidates form the largest group. Gender is missing for 23.5% of the records.

#### relevent_experience

In [ ]:
labeled_barplot(df, "relevent_experience")

Most candidates have relevant work experience (about 72%).

#### company_size

In [ ]:
labeled_barplot(df, "company_size")

Among the recorded company sizes, 50-99 employees is the most common group.

### Bivariate Analysis

In [ ]:
sns.pairplot(data=df, diag_kind="kde")
plt.show()

The pairplot compares the numerical columns. City development index shows a clearer relationship with the target than training hours.

In [ ]:
df["target"].replace(1, "yes", inplace=True)
df["target"].replace(0, "no", inplace=True)

#### target vs gender

In [ ]:
stacked_barplot(df, "gender", "target")

Among the recorded gender groups, the job-change share is slightly higher for females and the other group than for males.

In [ ]:
df["gender"] = df["gender"].replace(np.nan, "unknown")

In [ ]:
stacked_barplot(df, "gender", "target")

After adding an unknown category, the missing gender records can also be compared with the other groups.

#### target vs relevant_experience

In [ ]:
stacked_barplot(df, "relevent_experience", "target")

Candidates without relevant experience have a higher job-change share (33.8%) than those with experience (21.5%).

#### target vs company_type

In [ ]:
stacked_barplot(df, "company_type", "target")

The job-change percentage is highest in the recorded 'Other' company type, but this group has very few records.

#### target vs last_new_job

In [ ]:
stacked_barplot(df, "last_new_job", "target")

Candidates whose last job was 'never' have the highest job-change share (30.1%) among these groups.

#### target vs training_hours

In [ ]:
distribution_plot_wrt_target(df, "training_hours", "target")

Training hours look similar for both target groups, so they may not be a strong factor on their own.

#### target vs city_development_index

In [ ]:
distribution_plot_wrt_target(df, "city_development_index", "target")

The city development index has a negative relationship with job change; candidates from less developed cities show a higher job-change share.

## Data Preprocessing

In [ ]:
df.drop(["enrollee_id"], axis=1, inplace=True)

### Encoding categorical variables

In [ ]:
df["city"].nunique()

In [ ]:
df["city_development_index"].describe()

In [ ]:
df["city"] = pd.qcut(
    df["city_development_index"],
    q=[0, 0.25, 0.5, 1],
    labels=["Under_Developed", "Developing", "Developed"],
)

In [ ]:
df["city"].value_counts()

In [ ]:
data1 = df.copy()

In [ ]:
data1.head()

### Data Preparation for Modeling

In [ ]:
X = data1.drop(["target"], axis=1)
y = data1["target"].apply(lambda x: 1 if x == "yes" else 0)

In [ ]:
X.fillna(pd.NA,inplace=True)

                                               
X.replace(to_replace='unknown',value=pd.NA,inplace=True)

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)

                                                              

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=1, stratify=y_temp
)
print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
print("Number of rows in train data =", X_train.shape[0])
print("Number of rows in validation data =", X_val.shape[0])
print("Number of rows in test data =", X_test.shape[0])

In [ ]:
train_index = set(X_train.index)
val_index = set(X_val.index)
test_index = set(X_test.index)

### Missing Value Treatment

In [ ]:
X_train.isnull().sum()

In [ ]:
X_val.isnull().sum()

In [ ]:
X_test.isnull().sum()

In [ ]:
imp_mode = SimpleImputer(missing_values=pd.NA, strategy='most_frequent')

In [ ]:
X_train[:] = imp_mode.fit_transform(X_train[:])

                                                    
X_val[:] = imp_mode.transform(X_val[:])
X_test[:] = imp_mode.transform(X_test[:])

### Creating Dummy Variables

In [ ]:
final_df = pd.concat([X_train,X_val,X_test])

In [ ]:
final_df = pd.get_dummies(final_df, drop_first=True,dtype=int)
print(final_df.shape)

In [ ]:
X_train = final_df.loc[list(train_index)]

X_val = final_df.loc[list(val_index)]

X_test = final_df.loc[list(test_index)]

## Model Building

### Model evaluation criterion

In [ ]:
def model_performance_classification_sklearn(model, predictors, target):
    """
    Function to compute different metrics to check classification model performance

    model: classifier
    predictors: independent variables
    target: dependent variable
    """

                                                
    pred = model.predict(predictors)

    acc = accuracy_score(target, pred)                       
    recall = recall_score(target, pred)                     
    precision = precision_score(target, pred)                        
    f1 = f1_score(target, pred)                       

                                     
    df_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1": f1,},
        index=[0],
    )

    return df_perf

In [ ]:
def confusion_matrix_sklearn(model, predictors, target):
    """
    To plot the confusion_matrix with percentages

    model: classifier
    predictors: independent variables
    target: dependent variable
    """
    y_pred = model.predict(predictors)
    cm = confusion_matrix(target, y_pred)
    labels = np.asarray(
        [
            ["{0:0.0f}".format(item) + "\n{0:.2%}".format(item / cm.flatten().sum())]
            for item in cm.flatten()
        ]
    ).reshape(2, 2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt="")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")

### Model building with original data

In [ ]:
lr = LogisticRegression(random_state=1)
lr.fit(X_train, y_train)

In [ ]:
log_reg_model_train_perf = model_performance_classification_sklearn(
    lr, X_train, y_train
)
print("Training performance:")
log_reg_model_train_perf

In [ ]:
confusion_matrix_sklearn(lr, X_train, y_train)

In [ ]:
log_reg_model_val_perf = model_performance_classification_sklearn(lr, X_val, y_val)
print("Validation performance:")
log_reg_model_val_perf

In [ ]:
confusion_matrix_sklearn(lr, X_val, y_val)

In [ ]:
y_train.value_counts()

In [ ]:
scoring = "recall"
kfold = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=1
)                                       
cv_result_bfr = cross_val_score(
    estimator=lr, X=X_train, y=y_train, scoring=scoring, cv=kfold
)
cv_result_bfr

In [ ]:
plt.boxplot(cv_result_bfr)
plt.show()

### Model Building with oversampled data

In [ ]:
print("Before Oversampling, counts of label 'Yes': {}".format(sum(y_train == 1)))
print("Before Oversampling, counts of label 'No': {} \n".format(sum(y_train == 0)))

sm = SMOTE(
    sampling_strategy=1, k_neighbors=5, random_state=1
)                                              
X_train_over, y_train_over = sm.fit_resample(X_train, y_train)


print("After Oversampling, counts of label 'Yes': {}".format(sum(y_train_over == 1)))
print("After Oversampling, counts of label 'No': {} \n".format(sum(y_train_over == 0)))


print("After Oversampling, the shape of train_X: {}".format(X_train_over.shape))
print("After Oversampling, the shape of train_y: {} \n".format(y_train_over.shape))

In [ ]:
log_reg_over = LogisticRegression(random_state=1)

                                                                
log_reg_over.fit(X_train_over, y_train_over)

In [ ]:
log_reg_over_train_perf = model_performance_classification_sklearn(
    log_reg_over, X_train_over, y_train_over
)
print("Training performance:")
log_reg_over_train_perf

In [ ]:
log_reg_over_val_perf = model_performance_classification_sklearn(
    log_reg_over, X_val, y_val
)
print("validation performance:")
log_reg_over_val_perf

In [ ]:
confusion_matrix_sklearn(log_reg_over, X_val, y_val)

In [ ]:
scoring = "recall"
kfold = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=1
)                                       
cv_result_over = cross_val_score(
    estimator=log_reg_over, X=X_train_over, y=y_train_over, scoring=scoring, cv=kfold
)
cv_result_over

In [ ]:
plt.boxplot(cv_result_over)
plt.show()

### Regularization

In [ ]:
lr_estimator = LogisticRegression(random_state=1, solver="saga")

                                   
parameters = {"C": np.arange(0.1, 1.1, 0.1)}

                     
grid_obj = GridSearchCV(lr_estimator, parameters, scoring="recall")
grid_obj = grid_obj.fit(X_train_over, y_train_over)

                                                   
lr_estimator = grid_obj.best_estimator_

                                     
lr_estimator.fit(X_train_over, y_train_over)

In [ ]:
log_reg_reg_train_perf = model_performance_classification_sklearn(
    lr_estimator, X_train_over, y_train_over
)
print("Training performance:")
log_reg_reg_train_perf

In [ ]:
log_reg_reg_val_perf = model_performance_classification_sklearn(
    lr_estimator, X_val, y_val
)
print("Validation performance:")
log_reg_reg_val_perf

### Model Building with undersampled data

In [ ]:
rus = RandomUnderSampler(random_state=1, sampling_strategy=1)
X_train_un, y_train_un = rus.fit_resample(X_train, y_train)


print("Before UnderSampling, counts of label '1': {}".format(sum(y_train == 1)))
print("Before UnderSampling, counts of label '0': {} \n".format(sum(y_train == 0)))


print("After UnderSampling, counts of label '1': {}".format(sum(y_train_un == 1)))
print("After UnderSampling, counts of label '0': {} \n".format(sum(y_train_un == 0)))


print("After UnderSampling, the shape of train_X: {}".format(X_train_un.shape))
print("After UnderSampling, the shape of train_y: {} \n".format(y_train_un.shape))

In [ ]:
log_reg_under = LogisticRegression(random_state=1)
log_reg_under.fit(X_train_un, y_train_un)

In [ ]:
log_reg_under_train_perf = model_performance_classification_sklearn(
    log_reg_under, X_train_un, y_train_un
)
print("Training performance:")
log_reg_under_train_perf

In [ ]:
log_reg_under_val_perf = model_performance_classification_sklearn(
    log_reg_under, X_val, y_val
)
print("Validation performance:")
log_reg_under_val_perf

In [ ]:
scoring = "recall"
kfold = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=1
)                                       
cv_result_under = cross_val_score(
    estimator=log_reg_under, X=X_train_un, y=y_train_un, scoring=scoring, cv=kfold
)
cv_result_under

In [ ]:
plt.boxplot(cv_result_under)
plt.show()

## Model Performance Comparison and Final Model Selection

In [ ]:
models_train_comp_df = pd.concat(
    [
        log_reg_model_train_perf.T,
        log_reg_over_train_perf.T,
        log_reg_reg_train_perf.T,
        log_reg_under_train_perf.T,
    ],
    axis=1,
)
models_train_comp_df.columns = [
    "Logistic Regression",
    "Logistic Regression with oversampled data",
    "Regularised Logistic Regression",
    "Logistic Regression with undersampled data",
]
print("Training performance comparison:")
models_train_comp_df

In [ ]:
models_train_comp_df = pd.concat(
    [
        log_reg_model_val_perf.T,
        log_reg_over_val_perf.T,
        log_reg_reg_val_perf.T,
        log_reg_under_val_perf.T,
    ],
    axis=1,
)
models_train_comp_df.columns = [
    "Logistic Regression",
    "Logistic Regression with oversampled data",
    "Regularised Logistic Regression",
    "Logistic Regression with undersampled data",
]
print("Validation performance comparison:")
models_train_comp_df

### Test Performance

In [ ]:
log_reg_under_test_perf = model_performance_classification_sklearn(
    log_reg_under, X_test, y_test
)
print("Test performance:")
log_reg_under_test_perf

In [ ]:
confusion_matrix_sklearn(log_reg_under, X_test, y_test)

### Coefficient Interpretation

In [ ]:
coef_df = pd.DataFrame(
    np.append(log_reg_under.coef_, log_reg_under.intercept_),
    index=X_train.columns.tolist() + ["Intercept"],
    columns=["Coefficients"],
)
coef_df.T

In [ ]:
odds = np.exp(log_reg_under.coef_[0])

                               
perc_change_odds = (np.exp(log_reg_under.coef_[0]) - 1) * 100

                                                  
pd.set_option("display.max_columns", None)

                                
odds_df = pd.DataFrame({"Odds": odds, "Change_odd%": perc_change_odds}, index=X_train.columns).T

odds_df

## Conclusions and Business Recommendations

#### Note: The aim of this case study is to cover the concepts of Week 1 in Model Tuning course. More predictive models and hyperparameter tuning can be tried to improve the results.

## Appendix: Detailed Exploratory Data Analysis (EDA)

### Univariate Analysis

#### enrolled_university

In [ ]:
labeled_barplot(df, "enrolled_university")

Most candidates were not enrolled in a university (73.6%).

#### education_level

In [ ]:
labeled_barplot(df, "education_level")

Graduates are the largest education group, making up about 62% of the data.

#### major_discipline

In [ ]:
labeled_barplot(df, "major_discipline")

STEM is the most common major discipline, covering about 88.7% of recorded entries.

#### company_type

In [ ]:
labeled_barplot(df, "company_type")

Private companies are the most common company type among the recorded values.

#### last_new_job

In [ ]:
labeled_barplot(df, "last_new_job")

The most common gap between the current and previous job is 1 year.

### Bivariate analysis

#### Correlation Check

In [ ]:
plt.figure(figsize=(15, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, vmin=-1, vmax=1, fmt=".2f", cmap="Spectral")
plt.show()

The heatmap shows a negative correlation between city development index and the target. Training hours have a very weak correlation with the target.

#### target vs city

In [ ]:
stacked_barplot(df, "city", "target")

The under-developed city group has a much higher job-change share (49.8%) than the other two groups.

#### target vs major_discipline

In [ ]:
stacked_barplot(df, "major_discipline", "target")

The job-change percentages are fairly close across the major discipline groups.

#### target vs experience

In [ ]:
stacked_barplot(df, "experience", "target")

Candidates with less than 1 year of experience have a higher job-change share than most experienced groups.

#### target vs company_size

In [ ]:
stacked_barplot(df, "company_size", "target")

Job-change percentages are fairly similar across company sizes.

#### target vs enrolled_university

In [ ]:
stacked_barplot(df, "enrolled_university", "target")

Full-time university students have a higher job-change share (38.1%) than the other enrollment groups.

#### target vs education_level

In [ ]:
stacked_barplot(df, "education_level", "target")

Graduates have a higher job-change share (28%) than the other education groups shown.

### To jump back to the EDA summary section, click here.

## Overall Summary
Most candidates in this dataset are not looking for a job change. City development index, relevant experience and university enrollment show useful differences between the two target groups. The target is imbalanced, so the models are also tested with oversampling and undersampling.